# Mistral Small Fine-Tuning via La Plateforme


## Install & Imports

In [ ]:
!pip install -q --upgrade mistralai datasets plotly scikit-learn
!pip install python-dotenv

import os, re, math, json, time, random
import numpy as np
import plotly.graph_objects as go

from pathlib import Path
from datasets import load_dataset
from mistralai import Mistral
from sklearn.metrics import mean_squared_error, r2_score

print("Ready.")


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

## Constants

In [ ]:
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "") 

if not MISTRAL_API_KEY or not MISTRAL_API_KEY.strip():
    raise ValueError(
        "MISTRAL_API_KEY is not set.\n"
        "Set it via: os.environ['MISTRAL_API_KEY'] = 'your_key_here'\n"
        "or paste directly into MISTRAL_API_KEY above."
    )

client = Mistral(
    api_key=MISTRAL_API_KEY,
    server_url="https://api.mistral.ai", 
)


DATA_USER        = "ed-donner"
TRAIN_DATASET    = f"{DATA_USER}/items_prompts_lite"   # 20 K — used for fine-tuning
EVAL_DATASET     = f"{DATA_USER}/items_prompts_full"   # 10 K test split — used for both evals

TRAIN_FILE_PATH  = "./mistral_train.jsonl"
VAL_FILE_PATH    = "./mistral_val.jsonl"
SUFFIX           = "price-items"                       
EPOCHS           = 1
LEARNING_RATE    = 1e-4
WARMUP_FRACTION  = 0.05

MAX_TOKENS       = 30
TEMPERATURE      = 0.0  
EVAL_SIZE        = 1000
EVAL_SEED        = 42

BASE_MODEL = "mistral-small-2501"

print(f"Base model   : {BASE_MODEL}")
print(f"Train data   : {TRAIN_DATASET}")
print(f"Eval data    : {EVAL_DATASET}")
print(f"Key in use: ...{MISTRAL_API_KEY[-6:]}")


## Load Datasets

In [ ]:
lite_dataset = load_dataset(TRAIN_DATASET)
train_data   = lite_dataset["train"]
val_data     = lite_dataset["val"]

full_test    = load_dataset(EVAL_DATASET)["test"]

print(f"Train (lite)  : {len(train_data):,} rows")
print(f"Val   (lite)  : {len(val_data):,} rows")
print(f"Test  (full)  : {len(full_test):,} rows")
print(f"\nSample prompt   : {train_data[0]['prompt'][:200]}")
print(f"Sample completion: {train_data[0]['completion']}")


## Inference Helpers

In [ ]:
def build_messages(item):
    """Convert a dataset item to Mistral chat messages list."""
    return [{"role": "user", "content": item["prompt"]}]

def extract_price(text: str):
    """Extract the first plausible USD price from generated text."""
    patterns = [
        r"\$[\d,]+\.?\d*",
        r"[\d,]+\.\d{2}",
        r"[\d,]+",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace("$", "").replace(",", "")
            try:
                price = float(price_str)
                if 0.01 <= price <= 100_000:
                    return price
            except ValueError:
                continue
    return None


# Auth-related substrings — abort immediately, do not retry
_AUTH_ERRORS = ("401", "403", "Unauthorized", "Forbidden", "Illegal header", "Bearer")

# Seconds to sleep between every request — keeps us under the RPM limit
REQUEST_THROTTLE = 1.0   # increase to 2.0 if 429s persist; but you are likely to get 429s at startup due to cold start and/or if your key is very new

def predict(model_id: str, item: dict, retries: int = 5) -> float | None:
    """Call Mistral API and return extracted price.
    - Sleeps REQUEST_THROTTLE seconds before every call to stay under RPM limit.
    - Retries up to `retries` times on 429 with aggressive exponential back-off.
    - Raises immediately on auth errors.
    """
    time.sleep(REQUEST_THROTTLE)

    for attempt in range(retries):
        try:
            response = client.chat.complete(
                model       = model_id,
                messages    = build_messages(item),
                max_tokens  = MAX_TOKENS,
                temperature = TEMPERATURE,
            )
            return extract_price(response.choices[0].message.content)

        except Exception as e:
            err = str(e)

            if any(tag in err for tag in _AUTH_ERRORS):
                raise RuntimeError(
                    f"Authentication error — check your MISTRAL_API_KEY.\nDetail: {e}"
                )

            if "429" in err:
                # Aggressive back-off: 30s, 60s, 120s, 240s, 480s
                wait = 30 * (2 ** attempt)
                print(f"  Rate limited — waiting {wait}s (attempt {attempt+1}/{retries}) ...")
                time.sleep(wait)
            else:
                print(f"  API error (attempt {attempt+1}/{retries}): {e}")
                if attempt == retries - 1:
                    return None

    return None


## Evaluation Framework

In [ ]:
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"


class Tester:
    def __init__(self, model_id, data, title=None, size=None):
        self.model_id = model_id
        self.data     = data
        self.title    = title or model_id
        self.size     = size or len(data)
        self.titles   = []
        self.guesses  = []
        self.truths   = []
        self.errors   = []
        self.colors   = []

    def color_for(self, error, truth):
        pct = error / truth if truth else 0
        if pct < 0.2:  return "green"
        if pct < 0.4:  return "orange"
        return "red"

    def run(self):
        items = list(self.data)[:self.size]
        for i, item in enumerate(items):
            truth = float(item["completion"])
            pred  = predict(self.model_id, item)
            if pred is None:
                pred = truth * 2   # penalty for parse failure
            error = abs(pred - truth)
            color = self.color_for(error, truth)
            self.titles.append(item["prompt"][:60])
            self.guesses.append(pred)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            if (i + 1) % 50 == 0 or i == 0:
                mae = sum(self.errors) / len(self.errors)
                col = GREEN if color == "green" else (YELLOW if color == "orange" else RED)
                print(f"{col}[{i+1:>5}/{self.size}] Running MAE: ${mae:.2f}{RESET}")
        return self

    def chart(self):
        color_map = {"green": "green", "orange": "orange", "red": "red"}
        colors    = [color_map[c] for c in self.colors]
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=self.truths, y=self.guesses,
            mode="markers", name=self.title,
            marker=dict(color=colors, size=4, opacity=0.6),
            text=self.titles,
            hovertemplate="<b>%{text}</b><br>Actual: $%{x:.2f}<br>Predicted: $%{y:.2f}",
        ))
        max_val = max(max(self.truths), max(self.guesses))
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(dash="dash", color="gray", width=1),
            name="Perfect", hoverinfo="skip",
        ))
        mae = sum(self.errors) / len(self.errors)
        fig.update_layout(
            title=f"{self.title} | n={self.size:,} | MAE=${mae:.2f}",
            xaxis_title="Actual Price ($)", yaxis_title="Predicted Price ($)",
            width=800, height=550, template="plotly_white",
        )
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.show()

    def error_trend_chart(self):
        running_mae = [sum(self.errors[:i+1]) / (i+1) for i in range(len(self.errors))]
        n  = len(running_mae)
        ci = [1.96 * np.std(self.errors[:i+1]) / math.sqrt(i+1) for i in range(n)]
        fig = go.Figure([
            go.Scatter(x=list(range(n)), y=[m+c for m, c in zip(running_mae, ci)],
                       fill=None, mode="lines", line=dict(width=0), showlegend=False),
            go.Scatter(x=list(range(n)), y=[m-c for m, c in zip(running_mae, ci)],
                       fill="tonexty", mode="lines", line=dict(width=0),
                       fillcolor="rgba(0,100,255,0.15)", name="95% CI"),
            go.Scatter(x=list(range(n)), y=running_mae, mode="lines",
                       line=dict(color="royalblue", width=2), name="Running MAE"),
        ])
        fig.update_layout(
            title=f"Running MAE — {self.title}",
            xaxis_title="Item #", yaxis_title="MAE ($)",
            width=800, height=400, template="plotly_white",
        )
        fig.show()


def evaluate(model_id, data, title=None, size=None):
    t = Tester(model_id, data, title=title, size=size)
    t.run()
    t.chart()
    t.error_trend_chart()
    mae = sum(t.errors) / len(t.errors)
    mse = mean_squared_error(t.truths, t.guesses)
    r2  = r2_score(t.truths, t.guesses) * 100
    print(f"\n{'─'*40}")
    print(f"  MAE  : ${mae:.2f}")
    print(f"  MSE  : {mse:.0f}")
    print(f"  r²   : {r2:.1f}%")
    print(f"  Items: {len(t.errors):,}")
    print(f"{'─'*40}")
    return t

## Base Model Evaluation — Full Test Set

In [ ]:
base_tester = evaluate(
    BASE_MODEL,
    full_test,
    title = f"Base {BASE_MODEL}-1000 samples",
    size  = EVAL_SIZE, 
)


## Prepare Fine-Tuning Data


In [ ]:
def to_jsonl(dataset, path):
    """Write dataset to Mistral fine-tuning JSONL format."""
    with open(path, "w") as f:
        for item in dataset:
            record = {
                "messages": [
                    {"role": "user",      "content": item["prompt"]},
                    {"role": "assistant", "content": item["completion"]},
                ]
            }
            f.write(json.dumps(record) + "\n")
    size_mb = Path(path).stat().st_size / 1e6
    print(f"Written : {path}  ({len(dataset):,} rows, {size_mb:.1f} MB)")


to_jsonl(train_data, TRAIN_FILE_PATH)
to_jsonl(val_data,   VAL_FILE_PATH)


## Upload Training Files to Mistral

In [ ]:
with open(TRAIN_FILE_PATH, "rb") as f:
    train_file = client.files.upload(file={
        "file_name": "mistral_train.jsonl",
        "content": f,
    })

with open(VAL_FILE_PATH, "rb") as f:
    val_file = client.files.upload(file={
        "file_name": "mistral_val.jsonl",
        "content": f,
    })

print(f"Train file ID : {train_file.id}")
print(f"Val   file ID : {val_file.id}")

## Launch Fine-Tuning Job

In [ ]:
job = client.fine_tuning.jobs.create(
    model            = BASE_MODEL,
    training_files   = [{"file_id": train_file.id, "weight": 1}],
    validation_files = [val_file.id],
    suffix           = SUFFIX,
    hyperparameters  = {
        "training_steps": 10,
        "learning_rate": LEARNING_RATE,
    },
    auto_start       = True,
)

print(f"Job ID     : {job.id}")
print(f"Status     : {job.status}")
print(f"Model      : {job.model}")
print(f"\nMonitor at : https://console.mistral.ai/fine-tuning/{job.id}")
JOB_ID = job.id

## Monitor Fine-Tuning Job

In [ ]:
POLL_INTERVAL = 60   # seconds

while True:
    job = client.fine_tuning.jobs.get(job_id=JOB_ID)
    ts  = time.strftime("%H:%M:%S")
    print(f"[{ts}] Status: {job.status}")

    if job.status in ("SUCCESS", "FAILED", "CANCELLED"):
        break

    time.sleep(POLL_INTERVAL)

if job.status == "SUCCESS":
    FT_MODEL_ID = job.fine_tuned_model
    print(f"\n✅ Fine-tuning complete!")
    print(f"Fine-tuned model ID : {FT_MODEL_ID}")
else:
    print(f"\n❌ Job ended with status: {job.status}")
    FT_MODEL_ID = None


## Fine-Tuned Model Evaluation — Full Test Set

In [ ]:
# If resuming from a previous session, set FT_MODEL_ID manually:
# FT_MODEL_ID = "ft:mistral-small-latest:your-org:price-items:xxxxxxxx"
assert FT_MODEL_ID, "FT_MODEL_ID is not set — check job status in cell above."


In [ ]:
ft_tester = evaluate(
    FT_MODEL_ID,
    full_test,
    title = f"Fine-Tuned mistral-small-latest · Full test",
    size  = EVAL_SIZE,
)
